# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/budnikovanastya42-dot/Internship_week_1/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)
My lane is Refresh / Content Opportunity Scoring — Lane 2. Its defined output is "a ranked review queue with
scores, actions, and reason codes," so this is a ranking / scoring task: the question is
"which pages first?"

In [4]:
import pandas as pd
df = pd.read_csv("content_refresh_anonymized.csv")
print(f"{df.shape[0]} rows, {df.shape[1]} columns — one row per pseudonymized content item")
print(df['trend_direction'].value_counts())

30000 rows, 44 columns — one row per pseudonymized content item
trend_direction
down      16262
stable     5962
up         4388
new        2236
flat       1152
Name: count, dtype: int64


## 2. Target or proxy

Target: `is_declining_label = (trend_direction == "down")`.

- `1` = the page's `trend_direction` is `down` (last-30d impressions fell more than 20% vs the
  prior 30d)
- `0` = anything else (`up`, `stable`, `flat`, `new`)

This is a defined rule applied to an observed 30-vs-30-day comparison — not a separately
observed future outcome. Because the label is built from `trend_direction`/`trend_pct`, neither
column may ever be a model feature later — that would just be feeding the model the answer.

In [5]:
df['is_declining_label'] = (df['trend_direction'] == 'down').astype(int)
print(df['is_declining_label'].value_counts())
print(f"{df['is_declining_label'].mean()*100:.1f}% of rows are labeled declining (1)")


is_declining_label
1    16262
0    13738
Name: count, dtype: int64
54.2% of rows are labeled declining (1)


## 3. Success metric

Precision@50. I pick it because it saves a reviewer's time: they only ever look at a short
list, not all 30,000 rows, so the only thing worth measuring is that short list. I take the top 50 pages my ranking surfaces and ask: how many of them are actually pages that are already doing badly (declining) — worth a reviewer's attention — versus how many are there by mistake?

`precision@50 = (declining pages among my top 50) / 50`

What number means "good": clearly above pure chance (54.2% of all pages are already labeled
declining, so a careless top-50 could hit that by luck alone), and clearly above the hand-written
baseline rule once one exists. "Good" is relative to those two floors, not a fixed number I can
name yet — there's no ranking to plug into the formula at this framing stage
Cost of a wrong call: a genuinely declining page
stays unreviewed for another cycle while it keeps losing visibility.

In [7]:
def precision_at_k(ranked_labels, k=50):
    """ranked_labels: is_declining_label values, already sorted by a ranking score, best first.
    No such ranking exists yet — this just fixes the formula I'll apply once one does."""
    return ranked_labels[:k].mean()

## 4. The unit of analysis, as a real dataframe
One row = one pseudonymized content item (`content_id`), belonging to one client (`client_id`).

The `head(5)` below is just a structure preview — 5 rows to see what one row looks like.

In [8]:
cols = ['content_id', 'client_id', 'content_type', 'impressions_90d',
        'days_since_last_update', 'trend_direction', 'is_declining_label']
df[cols].head(5)

,content_id,client_id,content_type,impressions_90d,days_since_last_update,trend_direction,is_declining_label
0,content_304f48230142,client_f369cb89fc,keyword article,3803,20,down,1
1,content_a1fb4e703a9e,client_4e07408562,keyword article,15320,25,down,1
2,content_9aa793d4d895,client_7f2253d7e2,keyword article,12581,20,down,1
3,content_331d6c4de07b,client_19581e27de,keyword article,11751,22,stable,0
4,content_d99b7a2d90ca,client_3fdba35f04,keyword article,19140,14,down,1


## 5. Why ML beats a fixed rule here

In a rule, a human picks the
weights and thresholds by hand; a model learns them from data.

That difference only matters if it's actually true that one signal doesn't already tell you the others — if decline size already implied traffic volume, competition, and cpc, then one column would be enough on its own, and there'd be nothing left for a model to learn. So before claiming anything I have to check whether that's true, not just assume it: correlation
is the direct test for exactly this — it tells me whether knowing one column already tells me
another. High correlation would mean the signals overlap and a single rule column is enough. Low
correlation means they're independent, and the missing information genuinely isn't available from
any one column alone.

So I check it: severity (`-trend_pct`), `impressions_90d`, `competition`, `cpc`, `word_count`,
`days_since_last_update`, for declining pages only — how much does each pair actually move
together?

In [9]:
declining = df['trend_direction'].eq('down')
d = df.loc[declining].copy()
d['severity'] = -d['trend_pct']

# sanity-check the correlation two ways before trusting it
x = d['severity'].to_numpy()
y = d['impressions_90d'].to_numpy()
manual = ((x - x.mean()) * (y - y.mean())).sum() / len(x) / (x.std() * y.std())
print(f"severity vs impressions_90d — manual formula: {manual:.2f}, pandas .corr(): "
      f"{d['severity'].corr(d['impressions_90d']):.2f}")

cols = ['severity', 'impressions_90d', 'competition', 'cpc', 'word_count', 'days_since_last_update']
print()
print(d[cols].corr(numeric_only=True).round(2))

severity vs impressions_90d — manual formula: -0.15, pandas .corr(): -0.15

                        severity  impressions_90d  competition   cpc  \
severity                    1.00            -0.15         0.04  0.04   
impressions_90d            -0.15             1.00        -0.05 -0.03   
competition                 0.04            -0.05         1.00  0.34   
cpc                         0.04            -0.03         0.34  1.00   
word_count                 -0.18             0.16        -0.24 -0.12   
days_since_last_update     -0.08             0.06        -0.04  0.00   

                        word_count  days_since_last_update  
severity                     -0.18                   -0.08  
impressions_90d               0.16                    0.06  
competition                  -0.24                   -0.04  
cpc                          -0.12                    0.00  
word_count                    1.00                    0.29  
days_since_last_update        0.29                   

Only `competition` and `cpc` move together meaningfully (0.34 — expected, both come from
the same keyword-auction economics).
Six mostly-independent signals is a condition for ML to have room to help —
no single column stands in for the rest — but it isn't sufficient proof a rule fails.FlyRank's own
`baseline_refresh_score` is exactly that — 4 signals, weights 0.40/0.30/0.25/0.05, chosen by
hand. What a fixed rule can't easily do is find the right weights and interaction effects
empirically, or re-fit them as patterns shift — a human has to guess and then hold them fixed.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.